# Hornsby + Randwick DCPs: build ONE PixelRAG index on a free Colab GPU

One index holds both councils. Each council's pages go in their own folder (`pages/hornsby`, `pages/randwick`), so PixelRAG tags every page with its council and the app can search one council at a time (PixelRAG's `department` filter). One search server and one AI model serve both.

Size: 489 + 1,528 pages, about 8,200 page pieces to embed. On a T4 GPU expect roughly 30 to 90 minutes for the full build; keep this browser tab open while it runs (Colab disconnects idle tabs, and the build has no checkpoint).

**Before you start:** menu **Runtime > Change runtime type > T4 GPU**. Then run the cells top to bottom (Shift+Enter).

1. Cell 1 checks that a GPU is attached.
2. Cell 2 installs PixelRAG (about 2 minutes). Cell 3 checks the install.
3. Cell 4 asks you to upload BOTH PDFs from your laptop: `data/hornsby/raw/hdcp-2024.pdf` (42 MB) and `data/randwick/raw/randwick-dcp.pdf` (27 MB). Select both in the file picker.
4. Cell 5 renders the pages. It starts in **TEST mode** (5 pages per council, a few minutes in total). Run cells 5 to 8 once in test mode; if cell 7 shows both councils, set `ALL = True` in cell 5 and run cells 5 to 8 again.
5. Cell 6 builds the index on the GPU. Cell 7 checks both councils are in it.
6. Cell 8 downloads the small zip (about 80 MB for the full build). Save it into the project's `data` folder and tell Claude.


In [ ]:
# 1. Is a GPU attached?
import subprocess
gpu = subprocess.run(['nvidia-smi', '-L'], capture_output=True, text=True).stdout.strip()
print(gpu or 'NO GPU FOUND: Runtime > Change runtime type > T4 GPU, then run this cell again.')

In [ ]:
# 2. Install PixelRAG (index extra) and PyMuPDF (to render the PDF pages)
!pip -q install "pixelrag[index]==0.4.0" pymupdf

In [ ]:
# 3. Check the install. torch and torchvision must match each other, otherwise the embed step fails with
#    'operator torchvision::nms does not exist' (the same trap we hit on the laptop).
import importlib, torch, torchvision
print('torch', torch.__version__, '| torchvision', torchvision.__version__, '| cuda available:', torch.cuda.is_available())
try:
    from torchvision.ops import nms
    from transformers import AutoProcessor, Qwen3VLForConditionalGeneration
    print('OK: imports work')
except Exception as e:
    print('PROBLEM:', e)
    print('Fix: run the next cell, then Runtime > Restart session, then run cells 3 onwards again (skip cell 2).')

In [ ]:
# 3b. ONLY if cell 3 printed PROBLEM: reinstall torch and torchvision together so they match.
# !pip -q install --upgrade --force-reinstall torch torchvision

In [ ]:
# 4. Upload BOTH PDFs: hdcp-2024.pdf (Hornsby) and randwick-dcp.pdf (Randwick). Select both files in the picker.
from google.colab import files
uploaded = files.upload()
for name, data in uploaded.items():
    print('got', name, round(len(data) / 1e6, 1), 'MB')


In [ ]:
# 5. Render pages to PNG at 150 dpi into one folder per council, named p0017.png etc. (the same naming and
#    resolution as on the laptop, so page numbers in search results are the real PDF page numbers).
import os, shutil, fitz

ALL = False   # False = TEST mode (5 pages per council). True = every page of both PDFs.
PDFS = {'hornsby': 'hdcp-2024.pdf', 'randwick': 'randwick-dcp.pdf'}
TEST_PAGES = {'hornsby': [17, 37, 113, 440, 449], 'randwick': [1, 120, 360, 450, 700]}

shutil.rmtree('pages', ignore_errors=True)
for council, pdf in PDFS.items():
    if not os.path.exists(pdf):
        raise SystemExit(f'{pdf} was not uploaded: run cell 4 again and select both PDFs')
    doc = fitz.open(pdf)
    os.makedirs(f'pages/{council}')
    for p in (range(1, doc.page_count + 1) if ALL else TEST_PAGES[council]):
        doc[p - 1].get_pixmap(dpi=150).save(f'pages/{council}/p{p:04d}.png')
    print(council, len(os.listdir(f'pages/{council}')), 'pages rendered from', doc.page_count, '-page PDF; ALL =', ALL)


In [ ]:
# 6. Build the index on the GPU, from the folder that holds both council folders.
# IMPORTANT: --device auto (uses the GPU through PixelRAG's light embedder). --device cuda would switch to the
# vLLM/sglang backend, which is not installed here.
import time
t0 = time.time()
!pixelrag index build --source pages --source-type local --output index --device auto --force
print('index built in', round(time.time() - t0), 'seconds')


In [ ]:
# 7. Check: every page must be tagged with its council (the 'department').
import json, collections
arts = json.load(open('index/articles.json'))
print(collections.Counter(a.get('department', '') for a in arts))
print(json.load(open('index/summary.json')))


In [ ]:
# 8. Package ONLY the small files (search needs the vectors and page mapping, not the tile images) and download.
import pathlib, zipfile
OUT = 'dcp_index_combined_' + ('full' if ALL else 'test') + '.zip'
keep = {'index.faiss', 'metadata.npz', 'articles.json', 'summary.json', 'tiles.json', 'chunks.json'}
with zipfile.ZipFile(OUT, 'w', zipfile.ZIP_DEFLATED) as z:
    for f in pathlib.Path('index').rglob('*'):
        if f.is_file() and f.name in keep:
            z.write(f, f.relative_to('index'))
print(OUT, round(os.path.getsize(OUT) / 1e6, 1), 'MB')
files.download(OUT)
